# Lab 08: Evaluation & LLM-as-Judge

**Exam Domain:** Evaluation and Monitoring (12%)  
**Time:** ~40 minutes | **Cost:** ~$2–3

---

## Overview

In this lab you will build a complete evaluation pipeline for a Databricks-hosted LLM agent:

1. **Create an evaluation dataset** — question / expected-response pairs that represent the ground truth for your agent.
2. **Run built-in MLflow metrics** — `mlflow.evaluate` with `model_type="databricks-agent"` to get token usage, latency, and automatic quality scores.
3. **Define a custom LLM-as-a-Judge rubric** — use `mlflow.metrics.genai.make_genai_metric` to score citation quality on a 1–5 scale.
4. **Mosaic AI Agent Evaluation** — use `databricks.agents.evaluate` for relevance, groundedness, safety, and chunk-relevance metrics.
5. **Compare model versions** — loop over registered model versions and compare evaluation metrics to decide which version to promote.

By the end of the lab you will understand how to close the feedback loop between model deployment and evaluation, which is the core skill tested in the Evaluation and Monitoring exam domain.

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk mlflow databricks-agents --quiet
dbutils.library.restartPython()

> **Cost tip:** These labs run on **serverless compute** by default — no cluster setup needed. You only pay per-second of actual usage. The `COST_PROFILE` below lets you choose cheaper models if you're cost-sensitive.

In [ ]:
# === Cost Profile ===
# "budget"   — cheapest models, may have lower quality (~$0.50/lab)
# "balanced" — good quality/cost ratio (~$1-2/lab)  [DEFAULT]
# "quality"  — best models, highest cost (~$3-5/lab)
COST_PROFILE = "balanced"

_LLM_ENDPOINTS = {
    "budget":   "databricks-meta-llama-3-1-8b-instruct",
    "balanced": "databricks-meta-llama-3-3-70b-instruct",
    "quality":  "databricks-meta-llama-3-3-70b-instruct",
}
LLM_ENDPOINT = _LLM_ENDPOINTS[COST_PROFILE]

# ---------------------------------------------------------------------------
# Config — update these values to match your Databricks workspace
# ---------------------------------------------------------------------------
CATALOG    = "genai_lab_guide"                         # Unity Catalog catalog name
SCHEMA     = "default"                    # Schema / database name
MODEL_NAME = f"{CATALOG}.{SCHEMA}.arxiv_research_agent"  # Full 3-part model name in UC

# MLflow experiment — one experiment per lab so runs are grouped
import mlflow
username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/genai-lab-guide/lab-08-evaluation")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Model   : {MODEL_NAME}")

## A. Create Evaluation Dataset

An **evaluation dataset** is a table of ground-truth question/answer pairs.  
Each row must contain at minimum:

| Column | Purpose |
|---|---|
| `inputs` | The user question sent to the model |
| `expected_response` | The correct answer used by the judge model |

Optional columns (`context`, `retrieved_context`) are used by groundedness and chunk-relevance metrics.

> **Exam tip:** The exam may ask how many rows are needed for a statistically meaningful evaluation. A minimum of 25–50 question/answer pairs is the recommended starting point; fewer rows produce high-variance metrics.

In [ ]:
import pandas as pd

eval_data = pd.DataFrame([
    {
        "inputs": "What is the role of the attention mechanism in transformer models?",
        "expected_response": (
            "The attention mechanism allows each token in a sequence to dynamically "
            "weight the relevance of every other token, enabling the model to capture "
            "long-range dependencies without recurrence. Multi-head attention applies "
            "this process in parallel across multiple learned subspaces."
        ),
    },
    {
        "inputs": "Explain Low-Rank Adaptation (LoRA) and why it is used for fine-tuning LLMs.",
        "expected_response": (
            "LoRA injects trainable low-rank decomposition matrices (A and B, where rank r << d) "
            "into frozen pre-trained weight matrices. During fine-tuning only the LoRA matrices "
            "are updated, reducing trainable parameters by orders of magnitude and making "
            "fine-tuning feasible on consumer hardware without degrading base model quality."
        ),
    },
    {
        "inputs": "What is chain-of-thought prompting and when should it be used?",
        "expected_response": (
            "Chain-of-thought (CoT) prompting asks the model to produce intermediate "
            "reasoning steps before giving a final answer. It should be used for tasks "
            "requiring multi-step logic, arithmetic, or symbolic reasoning. CoT improves "
            "accuracy on complex tasks but increases output token count and latency."
        ),
    },
    {
        "inputs": "How does Retrieval-Augmented Generation (RAG) reduce hallucination in LLMs?",
        "expected_response": (
            "RAG grounds the LLM's response in retrieved documents by injecting relevant "
            "chunks into the context window at inference time. Because the model is "
            "constrained to cite the supplied context rather than relying purely on "
            "parametric memory, factual hallucinations are significantly reduced — "
            "especially for domain-specific or time-sensitive information."
        ),
    },
    {
        "inputs": "What is Constitutional AI and how does it differ from RLHF?",
        "expected_response": (
            "Constitutional AI (CAI) trains a model to critique and revise its own outputs "
            "according to a set of written principles (the 'constitution'), using AI "
            "feedback rather than human labellers for the RL signal. RLHF uses human "
            "preference labels to train a reward model. CAI reduces the human labelling "
            "bottleneck but requires careful constitution design to avoid encoding "
            "undesired values."
        ),
    },
])

print(f"Eval dataset: {len(eval_data)} rows")
display(eval_data)

## B. Run Evaluation with Built-in Metrics

`mlflow.evaluate` with `model_type="databricks-agent"` automatically computes:

| Metric | What it measures |
|---|---|
| `response/latency_seconds` | Wall-clock time per request |
| `token_count` | Total input + output tokens |
| `toxicity` | Proportion of responses flagged as toxic |
| `relevance_to_query` | 1–5 score: does the response answer the question? |
| `groundedness` | 1–5 score: is the response supported by retrieved context? |

> **Exam tip:** `model_type="databricks-agent"` activates the Databricks-specific judge model automatically. You do not need to specify a judge model URI when using this model type.

In [ ]:
import mlflow

MODEL_VERSION = 1
model_uri = f"models:/{MODEL_NAME}/{MODEL_VERSION}"

with mlflow.start_run(run_name=f"eval-builtin-v{MODEL_VERSION}") as run:
    eval_results = mlflow.evaluate(
        model=model_uri,
        data=eval_data,
        targets="expected_response",
        model_type="databricks-agent",
    )

print("\n--- Aggregate Metrics ---")
for metric, value in eval_results.metrics.items():
    print(f"  {metric:<45} {value:.4f}")

print("\n--- Per-Row Results ---")
display(eval_results.tables["eval_results"])

## C. LLM-as-a-Judge with Custom Rubric

Built-in metrics measure general quality. For domain-specific dimensions — such as whether the agent's response cites sources correctly — you need a **custom judge**.

`mlflow.metrics.genai.make_genai_metric` creates a metric that:
1. Calls a judge LLM with a grading prompt you supply.
2. Extracts a numeric score from the judge's response.
3. Records the score (and the judge's rationale) for every eval row.

Key parameters:

| Parameter | Purpose |
|---|---|
| `name` | The metric column name in the results table |
| `definition` | Plain-English description of what is being measured |
| `grading_prompt` | Rubric that maps score values (1–5) to observable criteria |
| `examples` | Few-shot `EvaluationExample` objects that anchor the rubric |
| `model` | Judge model URI (e.g. `"endpoints:/databricks-meta-llama-3-1-70b-instruct"`) |
| `parameters` | Inference params for the judge (temperature, max_tokens) |

> **Exam tip:** The judge model used in `make_genai_metric` is a **separate** model from the one being evaluated. Typically a larger, more capable model is chosen as the judge.

In [ ]:
from mlflow.metrics.genai import make_genai_metric, EvaluationExample

# ---------------------------------------------------------------------------
# Define the citation_quality rubric (1 = no citations, 5 = perfect citations)
# ---------------------------------------------------------------------------
citation_quality_rubric = """\
Score 1: The response makes factual claims with no citations or source references.
Score 2: The response mentions sources vaguely (e.g. "according to research") but does not name them.
Score 3: The response names at least one specific source but omits key attribution details (author, year, or paper title).
Score 4: The response cites specific papers or sources with enough detail to locate them, but minor gaps remain.
Score 5: Every factual claim is supported by a clearly attributed, locatable source (paper title + authors or DOI).
"""

# Few-shot anchors — these help the judge model calibrate its scores
citation_examples = [
    EvaluationExample(
        input="What is the transformer architecture?",
        output=(
            "The transformer was introduced in 'Attention Is All You Need' "
            "(Vaswani et al., 2017, NeurIPS). It replaces recurrence with "
            "multi-head self-attention and position-wise feed-forward layers."
        ),
        score=5,
        justification="Paper title, authors, year, and venue are all present.",
    ),
    EvaluationExample(
        input="What is LoRA?",
        output=(
            "LoRA reduces fine-tuning parameters by injecting low-rank matrices. "
            "Studies have shown this approach is parameter-efficient."
        ),
        score=2,
        justification="No paper, author, or year is named despite a factual claim.",
    ),
]

citation_metric = make_genai_metric(
    name="citation_quality",
    definition=(
        "Measures whether every factual claim in the response is supported by "
        "a specific, locatable citation (paper title, authors, and year)."
    ),
    grading_prompt=citation_quality_rubric,
    examples=citation_examples,
    model=f"endpoints:/{LLM_ENDPOINT}",
    parameters={"temperature": 0.0, "max_tokens": 256},
    greater_is_better=True,
)

# ---------------------------------------------------------------------------
# Run evaluation with the custom citation metric added
# ---------------------------------------------------------------------------
with mlflow.start_run(run_name=f"eval-citation-judge-v{MODEL_VERSION}") as run:
    eval_results_custom = mlflow.evaluate(
        model=model_uri,
        data=eval_data,
        targets="expected_response",
        model_type="databricks-agent",
        extra_metrics=[citation_metric],
    )

print("\n--- Aggregate Metrics (with custom judge) ---")
for metric, value in eval_results_custom.metrics.items():
    print(f"  {metric:<45} {value:.4f}")

print("\n--- Per-Row Results with citation_quality ---")
cols = ["inputs", "outputs", "citation_quality/score", "citation_quality/justification"]
display(eval_results_custom.tables["eval_results"][[c for c in cols if c in eval_results_custom.tables["eval_results"].columns]])

## D. Mosaic AI Agent Evaluation

**Mosaic AI Agent Evaluation** (`databricks.agents.evaluate`) is optimised for RAG and agent pipelines. It provides four metrics not available in generic `mlflow.evaluate`:

| Metric | What it measures |
|---|---|
| `relevance` | Does the response answer the user's question? |
| `groundedness` | Is every claim in the response supported by the retrieved context? |
| `safety` | Does the response contain harmful, offensive, or policy-violating content? |
| `chunk_relevance` | Are the retrieved chunks relevant to the question? (retriever quality) |

`chunk_relevance` is unique to Agent Evaluation — it measures the **retriever** independently from the **generator**, enabling you to diagnose whether a quality problem is in retrieval or generation.

> **Exam tip:** Know which metrics target the *retriever* vs the *generator*. `chunk_relevance` targets the retriever. `groundedness` and `relevance` target the generator.

In [ ]:
from databricks.agents import evaluate as agent_evaluate

# Agent Evaluation expects the model_uri and an eval DataFrame.
# The DataFrame must have an 'inputs' column (dict or string).
# Optionally add 'expected_response' and 'retrieved_context' for richer metrics.

with mlflow.start_run(run_name=f"eval-agent-eval-v{MODEL_VERSION}") as run:
    agent_eval_results = agent_evaluate(
        model_uri,
        data=eval_data,
        metrics=["relevance", "groundedness", "safety", "chunk_relevance"],
    )

print("\n--- Mosaic AI Agent Evaluation Metrics ---")
for metric, value in agent_eval_results.metrics.items():
    print(f"  {metric:<45} {value}")

print("\n--- Per-Row Agent Eval Results ---")
display(agent_eval_results.tables["eval_results"])

## E. Compare Model Versions

Evaluation is most valuable when it drives a **version comparison** — which registered model version should be promoted to the `@Champion` alias?

The pattern below:
1. Loops over two candidate versions.
2. Runs `mlflow.evaluate` for each.
3. Prints a side-by-side comparison of the key metrics.

In production you would automate this in a CI/CD job and use the Databricks SDK to update the model alias programmatically.

> **Exam tip:** The exam tests whether you know how to use model aliases (`@Champion`, `@Challenger`) to control which version is served in production. Evaluation results — not intuition — should drive alias updates.

In [ ]:
import mlflow

VERSIONS_TO_COMPARE = [1, 2]  # update to the versions registered in your workspace

comparison = {}  # version -> metrics dict

for version in VERSIONS_TO_COMPARE:
    uri = f"models:/{MODEL_NAME}/{version}"
    print(f"\nEvaluating version {version} — {uri}")

    with mlflow.start_run(run_name=f"eval-compare-v{version}") as run:
        results = mlflow.evaluate(
            model=uri,
            data=eval_data,
            targets="expected_response",
            model_type="databricks-agent",
        )
    comparison[version] = results.metrics

# ---------------------------------------------------------------------------
# Print side-by-side comparison
# ---------------------------------------------------------------------------
KEY_METRICS = [
    "relevance_to_query/mean",
    "groundedness/mean",
    "toxicity/ratio",
    "latency/mean",
    "token_count/mean",
]

print(f"\n{'Metric':<45}" + "".join(f"  v{v:<8}" for v in VERSIONS_TO_COMPARE))
print("-" * (45 + 12 * len(VERSIONS_TO_COMPARE)))
for metric in KEY_METRICS:
    row = f"{metric:<45}"
    for version in VERSIONS_TO_COMPARE:
        val = comparison[version].get(metric, float("nan"))
        row += f"  {val:<10.4f}"
    print(row)

# ---------------------------------------------------------------------------
# Optionally promote the best version to @Champion alias
# ---------------------------------------------------------------------------
best_version = max(
    VERSIONS_TO_COMPARE,
    key=lambda v: comparison[v].get("relevance_to_query/mean", 0),
)
print(f"\nRecommended version to promote: v{best_version}")

# Uncomment to promote programmatically via the Databricks SDK:
# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
# w.registered_models.set_alias(full_name=MODEL_NAME, alias="Champion", version_num=best_version)
# print(f"Alias @Champion updated to version {best_version}")

## Key Takeaways

| Concept | What you did | Why it matters |
|---|---|---|
| **Evaluation dataset** | Created 5 question/expected-response pairs | Ground truth is the foundation of every evaluation pipeline |
| **Built-in metrics** | `mlflow.evaluate` with `model_type="databricks-agent"` | Free metrics (relevance, groundedness, toxicity, latency) with zero extra config |
| **LLM-as-a-Judge** | `make_genai_metric` with a 1–5 citation rubric | Enables domain-specific, interpretable scoring that built-in metrics cannot capture |
| **Agent Evaluation** | `databricks.agents.evaluate` for retriever + generator metrics | `chunk_relevance` isolates retriever quality; essential for diagnosing RAG failures |
| **Version comparison** | Loop over versions, print side-by-side metrics | Data-driven alias promotion — the right way to decide which version goes to production |

---

### Next Steps

- Proceed to **Lab 09: Deployment & Model Serving** to learn how to serve the version you just evaluated.

---

> **Exam Domain Coverage:** This lab covers the full *Evaluation and Monitoring* domain (12% of exam).
> Key exam topics: evaluation dataset structure, built-in vs custom metrics, judge model selection,
> `chunk_relevance` vs `groundedness`, and version comparison patterns.

## Architecture Diagram

![Architecture Diagram](../assets/diagrams/lab-08-evaluation-llm-judge.png)

## Key Concepts

| Concept | Definition |
|---|---|
| **Evaluation Dataset** | A table of question/expected-response pairs that serves as ground truth for automated evaluation. Minimum columns: `inputs`. Add `expected_response` for reference-based metrics. |
| **Groundedness** | A metric (1–5) measuring whether every factual claim in the model's response is supported by the retrieved context. A grounded response does not add information beyond what the context contains. |
| **Relevance** | A metric (1–5) measuring whether the model's response actually answers the user's question. A relevant response is on-topic and complete. |
| **Chunk Relevance** | A retriever-level metric (unique to Mosaic AI Agent Evaluation) that scores each retrieved chunk for relevance to the query. Enables independent diagnosis of retriever quality. |
| **Safety** | A binary or scored metric that flags responses containing harmful, offensive, or policy-violating content. Returned by `databricks.agents.evaluate` as part of the standard metric suite. |
| **LLM-as-a-Judge** | A pattern where a second, typically more capable LLM is used to evaluate the output of the primary model. The judge model receives the question, the response, and a rubric, and returns a score with justification. |
| **Custom Rubric** | A grading prompt passed to `make_genai_metric` that maps observable response characteristics to integer scores on a defined scale (e.g. 1–5). Rubrics encode domain expertise that generic metrics cannot capture. |
| **`make_genai_metric`** | The MLflow function (`mlflow.metrics.genai.make_genai_metric`) used to create a custom LLM-as-a-Judge metric. Accepts a rubric, few-shot examples, and a judge model URI. |
| **Agent Evaluation** | The `databricks.agents.evaluate` API — a Databricks-specific evaluation framework that understands agent trace structure and provides retriever-level metrics not available in standard `mlflow.evaluate`. |

## Exam Preparation

### Exam Practice Questions

**Q1.** A data scientist runs `mlflow.evaluate` on a RAG agent and observes that `groundedness/mean` is 4.8 but `relevance_to_query/mean` is 2.1. What does this combination most likely indicate?

- A) The retriever is returning irrelevant chunks; the generator is hallucinating.
- B) The generator is faithfully reproducing the retrieved context, but the retrieved context does not answer the user's question.
- C) The model is producing toxic outputs that are unrelated to the query.
- D) The evaluation dataset is too small to produce reliable metrics.

**Answer: B** — High groundedness means the generator is staying within the retrieved context (not hallucinating). Low relevance means the response does not answer the question. This indicates a retriever problem: the wrong chunks are being retrieved, so the generator has no relevant information to work with. The fix is to improve the retrieval layer (embedding model, chunking strategy, or vector search parameters), not the generator.

---

**Q2.** A developer wants to evaluate whether an LLM agent cites its sources precisely. None of the built-in `mlflow.evaluate` metrics measure citation quality. What is the correct approach?

- A) Write a custom Python function that parses the response for DOI patterns using regex.
- B) Use `mlflow.metrics.genai.make_genai_metric` with a rubric that defines citation criteria on a 1–5 scale.
- C) Switch to `databricks.agents.evaluate` and use the `chunk_relevance` metric.
- D) Use `mlflow.evaluate` with `model_type="question-answering"` and compare against the expected response.

**Answer: B** — `make_genai_metric` is the standard mechanism for creating domain-specific LLM-as-a-Judge metrics in MLflow. A rubric-based approach is more robust than regex (citation formats vary) and more cost-effective than a fully custom evaluation pipeline. Option A would miss non-DOI citations. Option C measures retriever quality, not citation quality. Option D produces similarity scores, not citation scores.

---

**Q3.** Which evaluation framework should be used to independently measure the quality of the *retriever* component in a Databricks RAG pipeline?

- A) `mlflow.evaluate` with `model_type="databricks-agent"` and the `groundedness` metric.
- B) `mlflow.evaluate` with `model_type="question-answering"` and the `answer_similarity` metric.
- C) `databricks.agents.evaluate` with the `chunk_relevance` metric.
- D) A custom `make_genai_metric` that scores retrieved chunks for topic relevance.

**Answer: C** — `chunk_relevance` is the only standard Databricks metric that scores retrieved chunks independently. It is available exclusively through `databricks.agents.evaluate` because that API understands the agent's trace structure (which chunks were retrieved per request). `groundedness` measures whether the *generator* stayed within the context — it does not tell you whether the context itself was relevant.

---

**Q4.** An evaluation dataset has 100 rows. A team is choosing a judge model for a custom `make_genai_metric`. Model A is the same model being evaluated. Model B is a larger, more capable model. Model C is a smaller, cheaper model. Which choice produces the most reliable scores?

- A) Model A — it understands its own output best.
- B) Model B — a more capable model produces more calibrated judgements.
- C) Model C — lower cost allows more evaluation runs.
- D) Any model can be used; the rubric quality matters more than the judge model.

**Answer: B** — The judge model should be equal to or more capable than the evaluated model. Using the evaluated model as its own judge (Model A) introduces a self-evaluation bias: the model tends to rate its own outputs highly regardless of actual quality. Model C would reduce cost but at the expense of judge calibration, especially for nuanced rubric dimensions. While rubric quality matters, judge model capability is also a critical factor.

---

**Q5.** A production MLflow experiment has two registered model versions. Version 1 has `relevance_to_query/mean = 3.8` and `latency/mean = 2.1s`. Version 2 has `relevance_to_query/mean = 4.2` and `latency/mean = 4.8s`. The team wants to promote the better version to `@Champion`. What is the correct Databricks SDK call, and what tradeoff must be considered?

- A) Call `w.registered_models.delete_version(version=1)` and register version 2 as the new version.
- B) Call `w.registered_models.set_alias(full_name=MODEL_NAME, alias="Champion", version_num=2)` and accept that latency doubles; verify SLA requirements are met.
- C) Call `mlflow.register_model` to create a new version combining both models.
- D) Update the `@Champion` alias to version 1 because lower latency is always preferred over higher relevance.

**Answer: B** — `set_alias` is the correct SDK method. Updating an alias is a non-destructive operation — version 1 is not deleted and can be rolled back instantly by re-pointing the alias. The tradeoff is real: version 2's relevance improvement (+0.4 points, ~10%) comes at a latency cost (+2.7s, ~130%). Whether this tradeoff is acceptable depends on the application's SLA. Option A is destructive and irreversible. Option C is not a valid MLflow operation. Option D prioritises a non-quality metric without justification.

## Cost Breakdown

| Component | Detail | Estimated Cost |
|---|---|---|
| Databricks Serverless compute | Notebook execution (~40 min DBU) | ~$0.50 |
| Built-in metrics (Section B) | 5 rows × judge model calls × 2 runs | ~$0.25–0.50 |
| Custom citation judge (Section C) | 5 rows × judge LLM calls (70B model) | ~$0.50–1.00 |
| Mosaic AI Agent Evaluation (Section D) | 5 rows × 4 metrics | ~$0.50–1.00 |
| Version comparison loop (Section E) | 5 rows × 2 versions × judge calls | ~$0.25–0.50 |
| **Total** | | **~$2–3** |

**Estimated time:** ~40 min | **Estimated cost:** ~$2–3

> The dominant cost driver is the LLM judge — specifically the citation judge (Section C) and Agent Evaluation (Section D), both of which call a 70B-parameter model per eval row per metric. Reducing the eval dataset to 5 rows (as in this lab) keeps costs manageable for learning. Production datasets of 50–200 rows will cost proportionally more.